In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
# Define the path of your folder containing 60 zips
zip_folder = Path(r"C:\Users\ASUS\Desktop\tennis_data")

# Read proper tables
GameInfo_df = pd.read_parquet(zip_folder / "pbp_all.parquet")

MatchTournamentInfo_df = pd.read_parquet(zip_folder / "tournament_all.parquet")

In [15]:
# Create data frames with desired columns and clean
# Tournament df:
MatchTournamentInfo_df = MatchTournamentInfo_df[["match_id","tournament_name"]]
MatchTournamentInfo_df.drop_duplicates(subset="match_id", inplace=True)

# Game df:
SetsPerMatch_df = GameInfo_df.groupby("match_id")["set_id"].max().reset_index(name = "number_of_sets")

SetsPerMatch_df["number_of_sets"] = SetsPerMatch_df["number_of_sets"].astype(int)

# Merge dfs:
result_df = SetsPerMatch_df.merge(MatchTournamentInfo_df, on="match_id", how= "inner")

# Create a column to highlight matches with 3 sets (no more sets available in dataset)
result_df["final_set"] = result_df["number_of_sets"] == 3

In [ ]:
# Create a final table to store total number of matches and matches with 3 sets in each tournament to calculate the most competitive one
competition = (result_df.groupby("tournament_name")
    .agg(
        total_matches=("match_id", "count"),
        three_set_matches=("final_set", "sum")
    ).reset_index())

competition["three_set_match_percentage"] = (competition["three_set_matches"] /competition["total_matches"] * 100)

competition = competition.sort_values("three_set_match_percentage",ascending=False)

In [ ]:
competition[competition["total_matches"] > 22]

In [ ]:
competition[competition["total_matches"] <= 22]